In [10]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Load and prepare data exactly as before
df = pd.read_csv("../data/Sample - Superstore.csv",
                 parse_dates=["Order Date", "Ship Date"],
                 encoding="latin-1")

df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["YearMonth"] = df["Order Date"].dt.to_period("M")
df["Profit_Margin"] = (df["Profit"] / df["Sales"] * 100).round(2)

# Rebuild anomaly detection pipeline
features = ["Sales", "Quantity", "Discount", "Profit"]
X = df[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
iso_forest = IsolationForest(contamination=0.01,
                             random_state=42,
                             n_estimators=100)
iso_forest.fit(X_scaled)
df["Anomaly"] = iso_forest.predict(X_scaled)

# Add this line -- computes the raw anomaly score for each row
df["Anomaly_Score"] = iso_forest.decision_function(X_scaled)

df["Anomaly_Type"] = "Normal"
df.loc[(df["Anomaly"] == -1) & (df["Profit"] > 0), "Anomaly_Type"] = "Opportunity"
df.loc[(df["Anomaly"] == -1) & (df["Profit"] <= 0), "Anomaly_Type"] = "Risk"

# Compute regional metrics
region_metrics = df.groupby("Region").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).round(2)
region_metrics["Profit_Margin"] = (
    region_metrics["Total_Profit"] /
    region_metrics["Total_Sales"] * 100
).round(2)

# Compute category metrics
category_metrics = df.groupby("Category").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).round(2)
category_metrics["Profit_Margin"] = (
    category_metrics["Total_Profit"] /
    category_metrics["Total_Sales"] * 100
).round(2)

# Anomaly counts
risk_counts = df[df["Anomaly_Type"] == "Risk"].groupby(
    "Region").size()
opportunity_counts = df[df["Anomaly_Type"] == "Opportunity"].groupby(
    "Region").size()

# Now generate text chunks -- each chunk is one piece of knowledge
knowledge_base = []

# Chunk type 1 -- one summary per region
for region, row in region_metrics.iterrows():
    risks = risk_counts.get(region, 0)
    opps = opportunity_counts.get(region, 0)
    chunk = (
        f"Region: {region}. "
        f"Total Sales: {row['Total_Sales']}. "
        f"Total Profit: {row['Total_Profit']}. "
        f"Profit Margin: {row['Profit_Margin']}%. "
        f"Risk anomalies detected: {risks}. "
        f"Opportunity anomalies detected: {opps}."
    )
    knowledge_base.append({"topic": f"region_{region}", "text": chunk})

# Chunk type 2 -- one summary per category
for category, row in category_metrics.iterrows():
    chunk = (
        f"Category: {category}. "
        f"Total Sales: {row['Total_Sales']}. "
        f"Total Profit: {row['Total_Profit']}. "
        f"Profit Margin: {row['Profit_Margin']}%."
    )
    knowledge_base.append({"topic": f"category_{category}", "text": chunk})

# Chunk type 3 -- overall business summary
total_sales = df["Sales"].sum().round(2)
total_profit = df["Profit"].sum().round(2)
overall_margin = (total_profit / total_sales * 100).round(2)
total_risk = len(df[df["Anomaly_Type"] == "Risk"])
total_opportunity = len(df[df["Anomaly_Type"] == "Opportunity"])

knowledge_base.append({
    "topic": "overall_summary",
    "text": (
        f"Overall business summary from 2014 to 2017. "
        f"Total transactions: {len(df)}. "
        f"Total sales: {total_sales}. "
        f"Total profit: {total_profit}. "
        f"Overall profit margin: {overall_margin}%. "
        f"Total risk anomalies: {total_risk}. "
        f"Total opportunity anomalies: {total_opportunity}."
    )
})

# Chunk type 4 -- most critical transaction
worst = df[df["Anomaly_Type"] == "Risk"].nsmallest(1, "Anomaly_Score").iloc[0]
knowledge_base.append({
    "topic": "worst_transaction",
    "text": (
        f"Most anomalous risk transaction detected. "
        f"Date: {worst['Order Date'].date()}. "
        f"Category: {worst['Category']}. "
        f"Region: {worst['Region']}. "
        f"Sales: {round(worst['Sales'], 2)}. "
        f"Discount: {worst['Discount']}. "
        f"Profit: {round(worst['Profit'], 2)}. "
        f"Anomaly score: {round(worst['Anomaly_Score'], 4)}."
    )
})

# Compute reference points for relative context
best_region = region_metrics["Profit_Margin"].idxmax()
worst_region = region_metrics["Profit_Margin"].idxmin()
best_margin = region_metrics.loc[best_region, "Profit_Margin"]
worst_margin = region_metrics.loc[worst_region, "Profit_Margin"]
avg_margin = region_metrics["Profit_Margin"].mean().round(2)

best_category = category_metrics["Profit_Margin"].idxmax()
worst_category = category_metrics["Profit_Margin"].idxmin()

# Clear old knowledge base and rebuild with richer context
knowledge_base.clear()

# Richer regional chunks with relative comparisons
for region, row in region_metrics.iterrows():
    risks = risk_counts.get(region, 0)
    opps = opportunity_counts.get(region, 0)

    # Add relative performance label
    if row["Profit_Margin"] == best_margin:
        performance = f"the best performing region with a margin of {row['Profit_Margin']}%"
    elif row["Profit_Margin"] == worst_margin:
        performance = f"the weakest performing region with a margin of {row['Profit_Margin']}%, below the business average of {avg_margin}%"
    else:
        performance = f"an average performing region with a margin of {row['Profit_Margin']}%"

    chunk = (
        f"{region} is {performance}. "
        f"It generated total sales of {row['Total_Sales']} "
        f"and total profit of {row['Total_Profit']}. "
        f"Risk anomalies detected: {risks} transactions flagged as financially abnormal. "
        f"Opportunity anomalies detected: {opps} transactions with unusually high profit. "
        f"Compared to the business average margin of {avg_margin}%, "
        f"this region is {'below' if row['Profit_Margin'] < avg_margin else 'above'} average."
    )
    knowledge_base.append({"topic": f"region_{region}", "text": chunk})

# Richer category chunks
for category, row in category_metrics.iterrows():
    if row["Profit_Margin"] == category_metrics["Profit_Margin"].max():
        cat_label = "the highest margin category"
    elif row["Profit_Margin"] == category_metrics["Profit_Margin"].min():
        cat_label = "the lowest margin category and a priority concern"
    else:
        cat_label = "a mid-range margin category"

    chunk = (
        f"{category} is {cat_label} with a profit margin of {row['Profit_Margin']}%. "
        f"It generated total sales of {row['Total_Sales']} "
        f"and total profit of {row['Total_Profit']}. "
        f"For context, the best category margin is "
        f"{category_metrics['Profit_Margin'].max()}% "
        f"and the worst is {category_metrics['Profit_Margin'].min()}%."
    )
    knowledge_base.append({"topic": f"category_{category}", "text": chunk})

# Richer overall summary
knowledge_base.append({
    "topic": "overall_summary",
    "text": (
        f"Overall business performance from 2014 to 2017. "
        f"Total transactions: {len(df)}. "
        f"Total sales: {total_sales}. "
        f"Total profit: {total_profit}. "
        f"Overall profit margin: {overall_margin}%. "
        f"The business shows year over year sales growth but with profit "
        f"dips in mid and post holiday periods. "
        f"Total risk anomalies detected: {total_risk} transactions. "
        f"Total opportunity anomalies detected: {total_opportunity} transactions. "
        f"Best performing region: {best_region} at {best_margin}% margin. "
        f"Worst performing region: {worst_region} at {worst_margin}% margin. "
        f"Most concerning category: {worst_category} due to lowest profit margin."
    )
})

# Richer worst transaction chunk
knowledge_base.append({
    "topic": "worst_transaction",
    "text": (
        f"The most anomalous risk transaction was detected on "
        f"{worst['Order Date'].date()} in the {worst['Region']} region "
        f"under the {worst['Category']} category. "
        f"The transaction had sales of {round(worst['Sales'], 2)} "
        f"with a discount of {worst['Discount']} applied, "
        f"resulting in a loss of {round(worst['Profit'], 2)}. "
        f"This is particularly concerning because it occurred during "
        f"one of the two months in the dataset where total business profit "
        f"was negative. The anomaly score was {round(worst['Anomaly_Score'], 4)}, "
        f"indicating it is far from normal transaction patterns."
    )
})

# Recompute embeddings with new richer chunks
texts = [chunk["text"] for chunk in knowledge_base]
embeddings = model.encode(texts)

print(f"Richer knowledge base created with {len(knowledge_base)} chunks.")
print(f"Embeddings recomputed: {embeddings.shape}")

print(f"Knowledge base created with {len(knowledge_base)} chunks.\n")
for chunk in knowledge_base:
    print(f"[{chunk['topic']}]")
    print(chunk["text"])
    print()

Richer knowledge base created with 9 chunks.
Embeddings recomputed: (9, 384)
Knowledge base created with 9 chunks.

[region_Central]
Central is the weakest performing region with a margin of 7.92%, below the business average of 12.07%. It generated total sales of 501239.89 and total profit of 39706.36. Risk anomalies detected: 10 transactions flagged as financially abnormal. Opportunity anomalies detected: 11 transactions with unusually high profit. Compared to the business average margin of 12.07%, this region is below average.

[region_East]
East is an average performing region with a margin of 13.48%. It generated total sales of 678781.24 and total profit of 91522.78. Risk anomalies detected: 12 transactions flagged as financially abnormal. Opportunity anomalies detected: 28 transactions with unusually high profit. Compared to the business average margin of 12.07%, this region is above average.

[region_South]
South is an average performing region with a margin of 11.93%. It generat

In [4]:
from sentence_transformers import SentenceTransformer

# Load a small but powerful local embedding model
# This runs entirely on your machine -- no API call needed
# all-MiniLM-L6-v2 is fast, lightweight, and good for semantic similarity
model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract just the text from each chunk
texts = [chunk["text"] for chunk in knowledge_base]

# Convert all texts to embeddings in one batch
# Each text becomes a vector of 384 numbers
embeddings = model.encode(texts)

print(f"Embedding matrix shape: {embeddings.shape}")
print(f"Each chunk is now represented by {embeddings.shape[1]} numbers")
print(f"\nSample -- first 5 numbers of chunk 1 embedding:")
print(embeddings[0][:5])

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 6335.43it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding matrix shape: (9, 384)
Each chunk is now represented by 384 numbers

Sample -- first 5 numbers of chunk 1 embedding:
[ 0.04248983  0.01944258 -0.0412029   0.00351419 -0.01440036]


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

def find_relevant_chunks(question, top_k=2):
    # Convert the question into an embedding using the same model
    # Same model means same 384-number space -- comparable to our chunks
    question_embedding = model.encode([question])

    # Compute cosine similarity between question and every chunk
    # Returns a score between 0 and 1 for each chunk
    # 1 means identical meaning, 0 means completely unrelated
    similarities = cosine_similarity(question_embedding, embeddings)[0]

    # Get indices of top_k most similar chunks
    top_indices = similarities.argsort()[::-1][:top_k]

    # Return the most relevant chunks with their similarity scores
    results = []
    for idx in top_indices:
        results.append({
            "topic": knowledge_base[idx]["topic"],
            "text": knowledge_base[idx]["text"],
            "similarity": round(similarities[idx], 4)
        })
    return results

# Test the retrieval with a sample question
test_question = "which region is underperforming"
results = find_relevant_chunks(test_question)

print(f"Question: {test_question}\n")
for r in results:
    print(f"Topic: {r['topic']} | Similarity: {r['similarity']}")
    print(f"Chunk: {r['text']}")
    print()

Question: which region is underperforming

Topic: region_South | Similarity: 0.4311000108718872
Chunk: Region: South. Total Sales: 391721.9. Total Profit: 46749.43. Profit Margin: 11.93%. Risk anomalies detected: 9. Opportunity anomalies detected: 12.

Topic: region_West | Similarity: 0.4077000021934509
Chunk: Region: West. Total Sales: 725457.82. Total Profit: 108418.45. Profit Margin: 14.94%. Risk anomalies detected: 1. Opportunity anomalies detected: 17.



In [ ]:
import anthropic

# Initialise the Anthropic client with your API key
client = anthropic.Anthropic(api_key="Your_Anthropic_API_Key")

def ask_sentineliq(question):
    # Step 1 -- retrieve the most relevant chunks for this question
    relevant_chunks = find_relevant_chunks(question, top_k=3)

    # Step 2 -- build context string from retrieved chunks
    # This is what gets sent to Claude instead of raw data
    context = "\n\n".join([chunk["text"] for chunk in relevant_chunks])

    # Step 3 -- build the prompt
    # We give Claude a role, the context, and the question
    # We explicitly tell it to only use the provided context
    # This prevents hallucination on data it has not seen
    prompt = f"""You are SentinelIQ, a business intelligence assistant.
You have access to the following business data summaries:

{context}

Answer the following question using only the information provided above.
Be concise, specific, and use actual numbers from the context.
If the context does not contain enough information, say so clearly.

Question: {question}"""

    # Step 4 -- call Claude API
    message = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    # Step 5 -- extract and return the response text
    return message.content[0].text

# Test with three different questions
questions = [
    "which region is underperforming and why",
    "which category has the worst profit margin",
    "are there any dangerous transactions I should know about"
]

for q in questions:
    print(f"Question: {q}")
    print(f"Answer: {ask_sentineliq(q)}")
    print("-" * 50)

Question: which region is underperforming and why
Answer: Based on the data provided, the **South region is underperforming** for the following reasons:

1. **Lowest profit margin**: 11.93% compared to West (14.94%) and East (13.48%)
2. **Significantly lower sales volume**: $391,721.9 versus West ($725,457.82) and East ($678,781.24)
3. **Lowest total profit**: $46,749.43 compared to West ($108,418.45) and East ($91,522.78)

The South region generates less than half the sales of the other two regions and has a profit margin that's 3 percentage points below the West and 1.5 percentage points below the East.
--------------------------------------------------
Question: which category has the worst profit margin
Answer: Based on the provided data, I cannot determine which category has the worst profit margin because only one category is included in the information - Technology with a 17.4% profit margin. To compare profit margins across categories, I would need data for multiple categories.

In [11]:
print("=" * 50)
print("     SENTINELIQ BUSINESS INTELLIGENCE ASSISTANT")
print("=" * 50)
print("Ask questions about your business data.")
print("Type 'exit' to quit.\n")

while True:
    # Get question from user
    question = input("Your question: ").strip()

    # Exit condition
    if question.lower() == "exit":
        print("SentinelIQ session ended.")
        break

    # Skip empty input
    if not question:
        continue

    # Get answer from RAG pipeline
    print("\nSentinelIQ: ", end="")
    answer = ask_sentineliq(question)
    print(answer)
    print()

     SENTINELIQ BUSINESS INTELLIGENCE ASSISTANT
Ask questions about your business data.
Type 'exit' to quit.



Your question:  what is the overall profit margin of the business



SentinelIQ: The overall profit margin of the business is 12.47%.



Your question:  which category should we be most worried about



SentinelIQ: Based on the provided information, we should be most worried about the **Furniture category**.

The most significant risk identified in the data is a Furniture transaction that resulted in a substantial loss of -$1,862.31 on January 28, 2015. This transaction had an anomaly score of -0.0956, indicating it was far from normal patterns, and occurred during one of only two months where the entire business had negative profit.

However, I should note that the context provided only mentions this single category example. A complete assessment would require data on anomaly patterns and performance metrics across all product categories.



Your question:  exit


SentinelIQ session ended.
